# Marketing Campaign - Customer Management Service

In [1]:
import warnings
warnings.filterwarnings('ignore')

#Install Packages
#!python3 -m pip install fastapi --break-system-packages
#!python3 -m pip install nest_asyncio --break-system-packages
#!pip install --upgrade "uvicorn[standard]" --break-system-packages

In [2]:
#Import packages
import nest_asyncio
import uvicorn
import asyncio
import tempfile
import numpy as np

from fastapi import FastAPI
from uvicorn import Config, Server
from pydantic import BaseModel
from hdfs import InsecureClient
from tensorflow.keras.models import load_model as keras_load_model

In [3]:
# Classes
class Customers(BaseModel):
    first_name: str
    last_name: str
    age: int
    marital_status:str
    default:int

In [4]:
# Function to pull model from HDFS, store locally and load using keras.
def load_model_from_hdfs(filename):
    try:
        # Connect to HDFS using insecureclient
        client = InsecureClient('http://localhost:9870', user='hduser')

        hdfs_path = f'/marketing_campaign/models/{filename}'

        # Create a temporary local file
        with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as temp_file:
            # Download model from HDFS
            client.download(hdfs_path, temp_file.name, overwrite=True)

        # Load using keras (avoid recursion)
        model = keras_load_model(temp_file.name)

        # Rebuild metrics
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        print("Model successful loaded from HDFS!")
        model.summary()
        return model
    except Exception as e:
        print("Unable to load model from HDFS")
        print("Error details:", e)

# Usage
model = load_model_from_hdfs('ann_model.h5')


Model successful loaded from HDFS!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,337 (24.75 KB)

 Trainable params: 6,337 (24.75 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
nest_asyncio.apply()

app = FastAPI()

@app.get("/healthcheck")
async def healthcheck():
    return {"status":"OK", "status_message":"Server is up and running!"}

@app.post("/customers")
async def add_customers(customer_details: Customers):
    
    print(customer_details.first_name)
    
    #Things to do:
    # load model
    # predict using customer data
    # store results in hdfs
    
    
    return 'received'

In [10]:
config = Config(app=app, host="0.0.0.0", port=8000, lifespan="on")
server = Server(config)

await server.serve()


INFO:     Started server process [4593]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Joel
INFO:     127.0.0.1:52826 - "POST /customers HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4593]


In [13]:
arr = np.array([
    -0.73381122, -0.13459548,  0.23209867,  1.00226721, -0.44712678, -0.05684597,
    -0.59997746, -0.51149955, -0.35768913, -0.46969501, -0.17774749, -0.17085019,
     1.81796973, -0.24199513, -0.19863902, -0.3009257 , -0.14385218, -0.46849732,
    -0.18721259, -1.20844293,  1.56271992, -1.03837182,  1.3815644 , -0.31020778,
    -0.15769393, -0.14346119, -0.18752093, -0.20586063, -0.18158511, -0.21726705,
    -0.21535504, -0.15948052, -0.12647725, -0.1950926 , -0.19893213, -0.19657689,
    -0.2084088 , -0.20471959, -0.15983568, -0.22689152,  3.81901479, -0.1905806 ,
    -0.24224471, -0.22292232, -0.16160093, -0.11877547, -0.1145009 , -0.16508052,
    -0.12016835, -0.12952342, -0.20097373, -0.20981329, -0.18284866, -0.14029721,
    -0.48247677, -0.07137436, -0.29362043, -0.21699475, -0.51082915, -0.16611166,
    -0.11783824, -0.47720445,  2.67500894, -0.14922724, -0.11689392
])

arr = arr.reshape(1, -1)

prediction = model.predict(arr)
print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
[[0.00940403]]


In [14]:
# Sample Curl Request

# curl -X POST http://localhost:8000/customers \
#   -H "Content-Type: application/json" \
#   -d '{"first_name":"Joel", "last_name":"Montuya", "age":18, "marital_status":"single","default":0}'


25/10/02 13:28:20 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from cctbigdata:35993 in 10000 milliseconds
